# Notebook 4 — Tracing a Metric to Its Source

**Goal:** when two endpoints disagree about something that sounds like the same
number, don't stop at "the API behaves this way" — trace the disagreement to the
exact query that produces each number. This is the kind of debugging a backend or
CI engineer does routinely: reproduce, hypothesize from what's public (docs), then
confirm against the actual implementation.

We'll do exactly that with a real discrepancy in this API, using short excerpts
from the actual FastAPI backend and Kubernetes/Helm import jobs that build this
service (the source repo itself is private, so the relevant snippets are
reproduced here rather than linked).


In [ ]:
import requests
import pandas as pd
import geopandas as gpd

BASE_URL = "http://149.165.154.170:30080"
DAM = "UT00221"  # Mountain Dell
TARGETS = ["hospitals", "railroads", "svi_tracts", "gap_status", "transportation"]


def get_json(path, **params):
    resp = requests.get(f"{BASE_URL}{path}", params=params, timeout=20)
    resp.raise_for_status()
    return resp.json()


summary = get_json("/risk/summary", damnumber=DAM, targets=TARGETS)
precomputed = summary["counts"]
precomputed


## Step 1 — Reproduce the discrepancy

For each target: pull the dam's zone geometry and the target's feature geometry
straight from the API, then do our *own* spatial join with `geopandas.sjoin` and
count the matches — no trusting the API's arithmetic, just its geometry.


In [ ]:
zone_gdf = gpd.GeoDataFrame.from_features(
    get_json("/risk/zone.geojson", damnumber=DAM)["features"], crs="EPSG:4326"
)

rows = []
for target in TARGETS:
    feats = get_json("/risk/features/multi.geojson", damnumber=DAM, targets=target)["features"]
    target_gdf = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")
    joined = gpd.sjoin(target_gdf, zone_gdf, predicate="intersects", how="inner")
    rows.append({"target": target, "precomputed": precomputed[target], "self_joined": len(joined)})

comparison = pd.DataFrame(rows)
comparison["match"] = comparison["precomputed"] == comparison["self_joined"]
comparison


Four out of five targets match exactly. But **`gap_status` doesn't**: `/risk/summary`
says 51, the features we can independently verify say 1. That gap is too large to be
rounding — two different things are being counted under the same name.

## Step 2 — Check the docs before guessing

Resist the urge to invent an explanation from behavior alone. Both endpoints publish
an OpenAPI description — read them side by side first.


In [ ]:
spec = requests.get(f"{BASE_URL}/openapi.json", timeout=10).json()

print("--- /risk/features/multi.geojson ---")
print(spec["paths"]["/risk/features/multi.geojson"]["get"]["description"])

print("\n--- /risk/summary ---")
print(spec["paths"]["/risk/summary"]["get"]["description"])


`/risk/features/multi.geojson`'s own docs say it plainly: *"GAP Status: Only GAP
Status 1 (full legal protection) and 2 (semi-protected) areas are returned."*
`/risk/summary`'s docs say nothing about GAP status at all — just "fast count-style
summary." That asymmetry is a real clue, but it isn't proof of anything about
`/risk/summary` specifically; it only tells us what the *other* endpoint filters on.
Let's confirm what we're actually looking at before drawing a conclusion.


In [ ]:
gap_feats = get_json("/risk/features/multi.geojson", damnumber=DAM, targets="gap_status")["features"]
print("raw feature count returned:", len(gap_feats))
gap_feats[0]["properties"].get("gap_sts")


The one feature we get back has `gap_sts = "2"` — consistent with the "1-2 only"
claim in the docs. But that still doesn't prove `/risk/summary`'s 51 includes *other*
GAP status levels (3, 4, or unset) rather than something else entirely — the public
docs simply don't say. This is exactly where you'd stop guessing and go read the
implementation. Here are the two relevant excerpts, reproduced from the backend
(`app/app/main.py`) and the Helm import job (`chart/geo-risk-largest/templates/precompute-job.yaml`).

**Excerpt 1 — how `/risk/summary` counts `gap_status` (`main.py`, `_build_counts_sql`):**

```python
# gap_status isn't one of the precomputed "risk_metrics_cache" columns, so it
# falls through to this default branch — a live, per-request query:
else:
    parts.append(
        sql.SQL(
            "COALESCE((SELECT COUNT(*) FROM {tbl} g "
            "WHERE ST_Intersects(g.geom, b.geom)), 0) AS {alias}"
        ).format(tbl=fq(DB_SCHEMA, tbl), alias=sql.Identifier(t))
    )
```

No status filter anywhere in this branch — it counts **every** row in `gis.gap_status`
that intersects the dam's zone, regardless of GAP status level.

**Excerpt 2 — how the `gap_status` features get cached (`precompute-job.yaml`):**

```sql
-- Building gis.geojson_features_cache, one INSERT per target:
INSERT INTO gis.geojson_features_cache (damnumber, target, features_geojson)
SELECT z.damnumber, 'gap_status',
       gis.build_geojson_features(z.damnumber, 'gap_status', 'gap_status', 'gap_sts IN (1, 2)')
FROM gis.inundation_zones_largest z;
```

That fourth argument, `'gap_sts IN (1, 2)'`, is a SQL filter baked into the query that
builds the cached GeoJSON — it's the *only* place this filter exists.


## The explanation

Nothing gets simplified or dissolved. There are simply **two different queries**
behind one label:

- `/risk/summary`'s `gap_status` = *every* intersecting protected-area record,
  any status level (1 through 4).
- `/risk/features/multi.geojson`'s `gap_status` = only the subset with status 1 or 2
  (the two most-protected categories) — a filter that exists in the caching job, not
  in the summary count.

So for Mountain Dell, 51 records intersect the zone at *some* GAP status, and
exactly 1 of those 51 happens to carry status 1 or 2. Both numbers are "correct" —
they're just answering different questions dressed up in the same field name.

**Lesson:** an API can be internally consistent in its *code* while still being
inconsistent in its *naming* — two endpoints reusing a variable name without reusing
its filter logic. Public docs got us halfway to the answer (they told us about one
side's filter); only the source told us the other side had none. That's a normal
day in backend work: check the docs, and know when "the docs don't say" means "go
read the code," not "assume it's the same."


## Exercise — is this always necessary?

Notebook 2 found that `/risk/metrics/filters` silently ignores `hospital_num_beds`.
Was that case actually a "go read the source" situation like this one, or could you
have learned the whole story from the public docs alone? Fetch and read
`/risk/metrics/filters`'s OpenAPI description below and decide — then compare your
answer to how much source-reading `gap_status` needed above.


In [ ]:
print(spec["paths"]["/risk/metrics/filters"]["post"]["description"])
